## Normalization - Keep Within Sequence Length

In [5]:
import json
import sentencepiece as spm
import os

# ============================
# CONFIG
# ============================
TOKENIZER_PATH = r"C:\Users\prash\Documents\AI\Major Project\FineTuning\new-tokenizers\bpe-16-updated.model"

XLSUM_DIR_COM = r"C:\Users\prash\Documents\AI\Major Project\FineTuning\nepali_XLSum_v2.0\Filtered"
XLSUM_DIR_NORM = r"C:\Users\prash\Documents\AI\Major Project\FineTuning\nepali_XLSum_v2.0\Normalized"
INPUT_JSONL = os.path.join(XLSUM_DIR_COM, "train_plus_val_update.jsonl")
OUTPUT_JSONL = os.path.join(XLSUM_DIR_NORM, "train_plus_val_norm_update.jsonl")

TOTAL_SEQ_LENGTH = 1024
TARGET_BUDGET = 100
SOURCE_BUDGET = TOTAL_SEQ_LENGTH - TARGET_BUDGET  # 924

PROMPT_TOKENS = 10
ARTICLE_LIMIT = 910  # fixed chunk size for text
HALF_LIMIT = ARTICLE_LIMIT // 2  # 455

# ============================
# LOAD TOKENIZER
# ============================
sp = spm.SentencePieceProcessor()
sp.load(TOKENIZER_PATH)
print(f"✓ Tokenizer loaded (vocab size: {sp.vocab_size()})")

# ============================
# PROCESS DATA
# ============================
output_samples = []
new_id = 1

with open(INPUT_JSONL, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if not line.strip():
            continue

        item = json.loads(line)

        # Use original id if exists, else fallback to loop index + 1
        original_id = item.get("id", i + 1)

        text = item["text"]
        summary = item["summary"]

        text_tokens = sp.encode(text)
        summary_tokens = sp.encode(summary)

        # ----------------------------
        # SUMMARY TRUNCATION
        # ----------------------------
        if len(summary_tokens) > TARGET_BUDGET:
            summary_tokens = summary_tokens[:TARGET_BUDGET]
            summary = sp.decode(summary_tokens)

        # ----------------------------
        # TEXT CHUNKING WITH CONTROLLED LOGIC
        # ----------------------------
        idx = 0
        total_tokens = len(text_tokens)

        if total_tokens <= ARTICLE_LIMIT:
            # single chunk
            chunk_tokens = [t for t in text_tokens if t != sp.unk_id()]
            chunk_text = sp.decode(chunk_tokens)
            output_samples.append({
                "id": new_id,
                "chunked": "no",
                "text": chunk_text,
                "summary": summary
            })
            new_id += 1
        else:
            # multiple chunks following fixed 910 / 455 logic
            while idx < total_tokens:
                remaining = total_tokens - idx

                if remaining >= ARTICLE_LIMIT:
                    chunk_tokens = text_tokens[idx: idx + ARTICLE_LIMIT]
                    idx += ARTICLE_LIMIT
                elif remaining >= HALF_LIMIT:
                    chunk_tokens = text_tokens[idx: idx + HALF_LIMIT]
                    idx += HALF_LIMIT
                else:
                    chunk_tokens = text_tokens[idx: idx + remaining]
                    idx = total_tokens

                # remove unknown tokens
                chunk_tokens = [t for t in chunk_tokens if t != sp.unk_id()]
                chunk_text = sp.decode(chunk_tokens)

                output_samples.append({
                    "id": new_id,
                    "chunked": f"yes_original{original_id}",
                    "text": chunk_text,
                    "summary": summary
                })
                new_id += 1

# ============================
# SAVE OUTPUT
# ============================
with open(OUTPUT_JSONL, "w", encoding="utf-8") as f:
    for sample in output_samples:
        f.write(json.dumps(sample, ensure_ascii=False) + "\n")

print(f"✓ Saved {len(output_samples)} samples to:")
print(f"  {OUTPUT_JSONL}")

✓ Tokenizer loaded (vocab size: 16384)
✓ Saved 7670 samples to:
  C:\Users\prash\Documents\AI\Major Project\FineTuning\nepali_XLSum_v2.0\Normalized\train_plus_val_norm_update.jsonl


### View Samples

In [8]:
import json
import sentencepiece as spm

# ---------------- CONFIG ----------------
# JSONL_PATH = r"C:\Users\prash\Documents\AI\Major Project\FineTuning\nepali_XLSum_v2.0\Edited\tfidf_textrank_sp910.jsonl"
JSONL_PATH = r"C:\Users\prash\Documents\AI\Major Project\FineTuning\nepali_XLSum_v2.0\Combined\train_plus_val.jsonl"
SP_MODEL_PATH = r"C:\Users\prash\Documents\AI\Major Project\FineTuning\new-tokenizers\bpe-16-updated.model"

SHOW_SAMPLE_COUNT = 100  # how many samples to show
# ---------------------------------------

# Load SentencePiece tokenizer
sp = spm.SentencePieceProcessor()
sp.load(SP_MODEL_PATH)

# Load JSONL
with open(JSONL_PATH, "r", encoding="utf-8") as f:
    samples = [json.loads(line) for line in f]

print(f"\nShowing token lengths for first {SHOW_SAMPLE_COUNT} samples from:\n{JSONL_PATH}\n")

for sample in samples[:SHOW_SAMPLE_COUNT]:
    text_tokens = len(sp.encode(sample["text"]))
    summary_tokens = len(sp.encode(sample["summary"]))

    print(f"Sample ID: {sample['id']}")
    print(f"  Text tokens:    {text_tokens}")
    print(f"  Summary tokens: {summary_tokens}\n")



Showing token lengths for first 100 samples from:
C:\Users\prash\Documents\AI\Major Project\FineTuning\nepali_XLSum_v2.0\Combined\train_plus_val.jsonl

Sample ID: news-55625366
  Text tokens:    1123
  Summary tokens: 41

Sample ID: news-55663409
  Text tokens:    1019
  Summary tokens: 40

Sample ID: news-48223117
  Text tokens:    1623
  Summary tokens: 47

Sample ID: news-39996152
  Text tokens:    1642
  Summary tokens: 19

Sample ID: news-52947762
  Text tokens:    74
  Summary tokens: 28

Sample ID: news-49577917
  Text tokens:    396
  Summary tokens: 51

Sample ID: news-49304830
  Text tokens:    1429
  Summary tokens: 66

Sample ID: news-56562115
  Text tokens:    1849
  Summary tokens: 25

Sample ID: news-47758058
  Text tokens:    1290
  Summary tokens: 10

Sample ID: news-56915667
  Text tokens:    88
  Summary tokens: 15

Sample ID: news-57292229
  Text tokens:    1878
  Summary tokens: 29

Sample ID: news-44516694
  Text tokens:    84
  Summary tokens: 39

Sample ID: new

## Filtering - Text < Summary

In [1]:
"""
Filter test entries where article token length >= summary token length
and save to a new JSONL file
"""

import json
import sentencepiece as spm

# =========================
# CONFIG
# =========================
TOKENIZER_PATH = r"C:\Users\prash\Documents\AI\Major Project\FineTuning\new-tokenizers\bpe-16-updated.model"
TEST_JSONL = r"C:\Users\prash\Documents\AI\Major Project\FineTuning\nepali_XLSum_v2.0\Edited\tfidf_textrank_sp910.jsonl"
FILTERED_JSONL = r"C:\Users\prash\Documents\AI\Major Project\FineTuning\nepali_XLSum_v2.0\Filtered\train_val_tfidf_filter.jsonl"

# =========================
# LOAD TOKENIZER
# =========================
sp = spm.SentencePieceProcessor()
sp.load(TOKENIZER_PATH)
print(f"✓ Tokenizer loaded (vocab size: {sp.vocab_size()})")

# =========================
# PROCESS TEST DATA
# =========================
filtered_entries = []
total_samples = 0
removed_count = 0

with open(TEST_JSONL, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        item = json.loads(line)
        total_samples += 1
        
        text_tokens = sp.encode(item['text'])
        summary_tokens = sp.encode(item.get('summary', ""))
        
        # Keep entries where text token length >= summary token length
        if len(text_tokens) >= len(summary_tokens):
            filtered_entries.append(item)
        else:
            removed_count += 1

# =========================
# SAVE FILTERED DATA
# =========================
with open(FILTERED_JSONL, 'w', encoding='utf-8') as f:
    for item in filtered_entries:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

# =========================
# PRINT RESULTS
# =========================
print(f"Total test samples: {total_samples}")
print(f"Entries removed (text shorter than summary): {removed_count}")
print(f"Entries kept: {len(filtered_entries)}")
print(f"✓ Filtered data saved to: {FILTERED_JSONL}")


✓ Tokenizer loaded (vocab size: 16384)
Total test samples: 6533
Entries removed (text shorter than summary): 0
Entries kept: 6533
✓ Filtered data saved to: C:\Users\prash\Documents\AI\Major Project\FineTuning\nepali_XLSum_v2.0\Filtered\train_val_tfidf_filter.jsonl


## CSV Check

In [3]:
import csv
import sentencepiece as spm

# ============================
# CONFIG
# ============================
TOKENIZER_PATH = r"C:\Users\prash\Documents\AI\Major Project\FineTuning\new-tokenizers\bpe-16-updated.model"
INPUT_CSV = r"C:\Users\prash\Documents\AI\Major Project\FineTuning\nepali_XLSum_v2.0\Filtered\filtered_train.csv"

# ============================
# LOAD TOKENIZER
# ============================
sp = spm.SentencePieceProcessor()
sp.load(TOKENIZER_PATH)
print(f"✓ Tokenizer loaded (vocab size: {sp.vocab_size()})\n")

# ============================
# READ CSV & PRINT INFO
# ============================
with open(INPUT_CSV, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)

    for i, row in enumerate(reader, start=1):
        summary = row["summary"]
        token_count = len(sp.encode(summary))

        print(f"Row {i}")
        print(f"  Summary token count: {token_count}\n")


✓ Tokenizer loaded (vocab size: 16384)

Row 1
  Summary token count: 10

Row 2
  Summary token count: 9

Row 3
  Summary token count: 10

Row 4
  Summary token count: 10

Row 5
  Summary token count: 10

Row 6
  Summary token count: 10

Row 7
  Summary token count: 10

Row 8
  Summary token count: 10

Row 9
  Summary token count: 9

Row 10
  Summary token count: 10

Row 11
  Summary token count: 11

Row 12
  Summary token count: 8

Row 13
  Summary token count: 10

Row 14
  Summary token count: 13

Row 15
  Summary token count: 10

Row 16
  Summary token count: 8

Row 17
  Summary token count: 9

Row 18
  Summary token count: 10

Row 19
  Summary token count: 8

Row 20
  Summary token count: 12

Row 21
  Summary token count: 10

Row 22
  Summary token count: 10

Row 23
  Summary token count: 10

Row 24
  Summary token count: 10

Row 25
  Summary token count: 10

Row 26
  Summary token count: 9

Row 27
  Summary token count: 9

Row 28
  Summary token count: 10

Row 29
  Summary token co

## Filter > 100 Summary Tokens

In [3]:
import json
import sentencepiece as spm

# ============================
# CONFIG
# ============================
TOKENIZER_PATH = r"C:\Users\prash\Documents\AI\Major Project\FineTuning\new-tokenizers\bpe-16-updated.model"
INPUT_JSONL = r"C:\Users\prash\Documents\AI\Major Project\FineTuning\nepali_XLSum_v2.0\Filtered\train_val_tfidf_filter.jsonl"
OUTPUT_JSONL = r"C:\Users\prash\Documents\AI\Major Project\FineTuning\nepali_XLSum_v2.0\Filtered\train_val_tfidf_filterv2.jsonl"

MAX_TOKENS = 100

# ============================
# LOAD TOKENIZER
# ============================
sp = spm.SentencePieceProcessor()
sp.load(TOKENIZER_PATH)
print(f"✓ Tokenizer loaded (vocab size: {sp.vocab_size()})\n")

# ============================
# FILTERING
# ============================
kept = 0
removed = 0
total_tokens_kept = 0

print(f"Filtering summaries with token length ≤ {MAX_TOKENS}...\n")

with open(INPUT_JSONL, "r", encoding="utf-8") as fin, \
     open(OUTPUT_JSONL, "w", encoding="utf-8") as fout:

    for line_number, line in enumerate(fin, start=1):
        if not line.strip():
            continue

        try:
            row = json.loads(line)
            summary_text = row.get("summary", "")

            token_count = len(sp.encode(summary_text))

            if token_count <= MAX_TOKENS:
                fout.write(json.dumps(row, ensure_ascii=False) + "\n")
                kept += 1
                total_tokens_kept += token_count
            else:
                removed += 1

        except json.JSONDecodeError:
            print(f"Warning: Could not decode JSON at line {line_number}")

# ============================
# FINAL STATISTICS
# ============================
print("=" * 50)
print(f"{'FILTERING SUMMARY':^50}")
print("=" * 50)
print(f"Total entries kept     : {kept}")
print(f"Total entries removed  : {removed}")

if kept > 0:
    print(f"Average tokens (kept)  : {total_tokens_kept / kept:.2f}")

print(f"Output written to      :\n{OUTPUT_JSONL}")
print("=" * 50)


✓ Tokenizer loaded (vocab size: 16384)

Filtering summaries with token length ≤ 100...

                FILTERING SUMMARY                 
Total entries kept     : 6528
Total entries removed  : 5
Average tokens (kept)  : 27.41
Output written to      :
C:\Users\prash\Documents\AI\Major Project\FineTuning\nepali_XLSum_v2.0\Filtered\train_val_tfidf_filterv2.jsonl


### TF-IDF and TextRank

In [5]:
import json
import numpy as np
import networkx as nx
import sentencepiece as spm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ---------------- CONFIG ----------------
INPUT_JSONL = r"C:\Users\prash\Documents\AI\Major Project\FineTuning\nepali_XLSum_v2.0\Combined\train_plus_val.jsonl"
OUTPUT_JSONL = r"C:\Users\prash\Documents\AI\Major Project\FineTuning\nepali_XLSum_v2.0\Edited\tfidf_textrank_sp910.jsonl"

SP_MODEL_PATH = r"C:\Users\prash\Documents\AI\Major Project\FineTuning\new-tokenizers\bpe-16-updated.model"

MAX_SAMPLES = 6533
MAX_SOURCE_TOKENS = 910

TFIDF_WEIGHT = 0.5
TEXTRANK_WEIGHT = 0.5
# ---------------------------------------

# -------- Load SentencePiece tokenizer --------
sp = spm.SentencePieceProcessor()
sp.load(SP_MODEL_PATH)


def count_tokens(text: str) -> int:
    return len(sp.encode(text, out_type=int))


# -------- Sentence utilities --------
def split_sentences(text):
    return [s.strip() for s in text.split("।") if len(s.strip()) > 5]


# -------- Scoring methods --------
def tfidf_scores(sentences):
    tfidf = TfidfVectorizer().fit_transform(sentences)
    return tfidf.sum(axis=1).A1


def textrank_scores(sentences):
    tfidf = TfidfVectorizer().fit_transform(sentences)
    sim_matrix = cosine_similarity(tfidf)
    np.fill_diagonal(sim_matrix, 0)

    graph = nx.from_numpy_array(sim_matrix)
    scores = nx.pagerank(graph)

    return np.array([scores[i] for i in range(len(sentences))])


# -------- Extraction with HARD token constraint --------
def extract_sentences(text, max_tokens):
    sentences = split_sentences(text)

    if not sentences:
        return ""

    tfidf = tfidf_scores(sentences)
    textrank = textrank_scores(sentences)

    combined_scores = TFIDF_WEIGHT * tfidf + TEXTRANK_WEIGHT * textrank

    ranked = sorted(
        zip(sentences, combined_scores),
        key=lambda x: x[1],
        reverse=True
    )

    selected = []

    for sent, _ in ranked:
        candidate = selected + [sent]
        candidate_text = "। ".join(candidate) + "।"

        # HARD check on final serialized text
        if count_tokens(candidate_text) > max_tokens:
            continue

        selected.append(sent)

    if not selected:
        return ""

    final_text = "। ".join(selected) + "।"

    # FINAL SAFETY GUARANTEE
    assert count_tokens(final_text) <= max_tokens

    return final_text


# -------- Main processing loop --------
def main():
    processed = 0

    with open(INPUT_JSONL, "r", encoding="utf-8") as fin, \
         open(OUTPUT_JSONL, "w", encoding="utf-8") as fout:

        for line in fin:
            if processed >= MAX_SAMPLES:
                break

            item = json.loads(line)

            # Only process if text exceeds MAX_SOURCE_TOKENS
            if count_tokens(item["text"]) > MAX_SOURCE_TOKENS:
                extracted_text = extract_sentences(
                    item["text"],
                    MAX_SOURCE_TOKENS
                )
            else:
                extracted_text = item["text"]  # keep as-is

            new_item = {
                "id": item["id"],
                "url": item["url"],
                "title": item["title"],
                "text": extracted_text,      # ≤ 910 SP tokens or original
                "summary": item["summary"]   # ORIGINAL summary
            }

            fout.write(json.dumps(new_item, ensure_ascii=False) + "\n")
            processed += 1

    print(f"Processed {processed} samples with conditional TF-IDF/TextRank extraction.")


if __name__ == "__main__":
    main()


Processed 6533 samples with conditional TF-IDF/TextRank extraction.
